In [0]:
from pyspark.sql.functions import (
    col,
    timestamp_diff,
    round as f_round,
    year,
    month,
    day,
    hour,
)
from nyc_taxi_trips_etl.utils.pipeline_params_parsing import (
    parse_and_expand_year_months,
)

year_months_to_extract = dbutils.widgets.get("year_months_to_extract")
catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id")

In [0]:
year_months = parse_and_expand_year_months(year_months_to_extract)

In [0]:
trips_df = spark.table(f"{catalog}.bronze.yellow_trip_data").where(
    f"pipeline_run_id = '{run_id}'"
)

In [0]:
trips_column_renamed = trips_df.select(
    col("VendorID").alias("vendor_id"),
    col("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime"),
    col("passenger_count"),
    col("trip_distance"),
    col("RatecodeID").alias("rate_code_id"),
    col("store_and_fwd_flag"),
    col("PULocationID").alias("pickup_location_id"),
    col("DOLocationID").alias("dropoff_location_id"),
    col("payment_type"),
    col("total_amount"),
    col("year_month"),
    col("pipeline_run_id"),
)

In [0]:
filtered_wrong_months_df = trips_column_renamed.where("""
    year_month = date_format(tpep_pickup_datetime, 'yyyy-MM')
""")

invalid_rows = trips_column_renamed.where("""
    year_month <> date_format(tpep_pickup_datetime, 'yyyy-MM')
""")

invalid_rows_cnt = invalid_rows.select("year_month").count()
print("Invalid_rows_count:", invalid_rows_cnt)

Invalid_rows_count: 268


In [0]:
deduplicated_df = spark.sql(
    """
    SELECT *
    FROM {filtered_df}
    QUALIFY COUNT(*) OVER(
        PARTITION BY year_month, vendor_id, tpep_pickup_datetime,
            tpep_dropoff_datetime, abs(total_amount)
        ) = 1
    """,
    filtered_df=filtered_wrong_months_df,
)

In [0]:
removed_invalid_rows_df = spark.sql(
    """
    SELECT *
    FROM {deduplicated_df}
    WHERE tpep_dropoff_datetime > tpep_pickup_datetime  -- positive duration only
    AND total_amount >= 0  -- positive amount only
    AND passenger_count > 0  -- at least one passenger
    AND trip_distance < 50 -- very unlikely that trip is longer than 50 miles.
    -- If it is, we can treat is as an outlier which should be filtered out
    AND trip_distance > 0
    """,
    deduplicated_df=deduplicated_df,
)

In [0]:
trip_speed_df = (
    removed_invalid_rows_df.withColumn(
        "trip_hours",
        f_round(
            timestamp_diff(
                "second",
                col("tpep_pickup_datetime"),
                col("tpep_dropoff_datetime"),
            )
            / 3600.0,
            3,
        ),
    )
    .where("trip_hours > 0")  # filter out trips with zero or negative duration
    .withColumn(
        "trip_speed_mph", f_round(col("trip_distance") / col("trip_hours"), 3)
    )
)

In [0]:
filtered_invalid_speed_and_duration_df = trip_speed_df.where(
    trip_speed_df.trip_speed_mph < 55
).where(trip_speed_df.trip_hours < 24)

In [0]:
selected_columns_df = filtered_invalid_speed_and_duration_df.select(
    col("vendor_id"),
    year(col("tpep_pickup_datetime")).alias("pickup_year"),
    month(col("tpep_pickup_datetime")).alias("pickup_month"),
    day(col("tpep_pickup_datetime")).alias("pickup_day"),
    hour(col("tpep_pickup_datetime")).alias("pickup_hour"),
    year(col("tpep_dropoff_datetime")).alias("dropoff_year"),
    month(col("tpep_dropoff_datetime")).alias("dropoff_month"),
    day(col("tpep_dropoff_datetime")).alias("dropoff_day"),
    hour(col("tpep_dropoff_datetime")).alias("dropoff_hour"),
    col("passenger_count"),
    col("trip_distance"),
    col("trip_hours"),
    col("trip_speed_mph"),
    col("rate_code_id"),
    col("pickup_location_id"),
    col("dropoff_location_id"),
    col("payment_type"),
    col("total_amount"),
    col("year_month"),
    col("pipeline_run_id"),
)

In [0]:
replace_where = "year_month in ({})".format(
    ",".join(f"'{ym}'" for ym in year_months)
)
(
    selected_columns_df.write.mode("overwrite")
    .option("replaceWhere", replace_where)
    .saveAsTable(f"{catalog}.silver.yellow_trip_data")
)